## 27-03-26


In [2]:
!pip install ollama

In [4]:
!pip install chromadb

   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/21.9 MB ? eta -:--:--
    --------------------------------------- 0.5/21.9 MB 837.4 kB/s eta 0:00:26
   - -------------------------------------- 0.8/21.9 MB 887.7 kB/s eta 0:00:24
   - -------------------------------------- 0.8/21.9 MB 887.7 kB/s eta 0:00:24
   -- ------------------------------------- 1.3/21.9 MB 1.0 MB/s eta 0:00:20
   --- ------------------------------------ 1.8/21.9 MB 1.3 MB/s eta 0:00:16
   ---- ----------------------------------- 2.6/21.9 MB 1.7 MB/s eta 0:00:12
   ---- ----------------------------------- 2.6/21.9 MB 1.7 MB/s eta 0:00:12
   ---- ----------------------------------- 2.6/21.9 MB 1.7 MB/s eta 0:00:12
   ---- ----------------------------------- 2.6/21.9 MB 1.7 MB/s eta 0:00:12
   ---- -------------

In [6]:
!pip install pypdf

In [10]:
!ollama pull llama3

pulling manifest â ‹ pulling manifest â ™ pulling manifest â ¹ pulling manifest â ¸ pulling manifest â ¼ pulling manifest â ´ pulling manifest â ¦ pulling manifest â § pulling manifest â ‡ pulling manifest â � pulling manifest â ‹ pulling manifest â ™ pulling manifest â ¹ pulling manifest â ¸ pulling manifest â ¼ pulling manifest â ´ pulling manifest â ¦ pulling manifest â § pulling manifest â ‡ pulling manifest â � pulling manifest 
pulling 6a0746a1ec1a: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 4.7 GB                         
pulling 4fa551d4f938: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  12 KB                         
pulling 8ab4849b038c: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  254 B                         
pulling 577073ffcc6c: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  110 B                         
pulling 3f8eb4da87fa: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ

In [ ]:
!ollama pull mxbai-embed-large

In [ ]:
import ollama
import chromadb
from pypdf import PdfReader

# --- CONFIGURATION ---
PDF_PATH = "ollama.pdf"
COLLECTION_NAME = "my_docs1"

# 1. Initialize Local Vector Database
client = chromadb.PersistentClient(path="./chroma_db1")
collection = client.get_or_create_collection(name=COLLECTION_NAME)

def ingest_pdf(path):
    """Reads a PDF, chunks it, and stores it in the vector DB."""
    print(f"--- Learning from {path} ---")
    reader = PdfReader(path)
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            # Generate embedding for the page content
            # We use mxbai-embed-large as it is excellent and fast
            response = ollama.embeddings(model="mxbai-embed-large", prompt=text)
            embedding = response["embedding"]
            
            # Store in ChromaDB
            collection.add(
                ids=[f"page_{i}"],
                embeddings=[embedding],
                documents=[text]
            )
    print("--- Knowledge Base Ready! ---")

def ask_rag_agent(question):
    """Retrieves relevant context and answers the question."""
    # Step 1: Embed the user's question
    query_embedding = ollama.embeddings(model="mxbai-embed-large", prompt=question)["embedding"]

    # Step 2: Retrieve the most relevant page from the DB
    results = collection.query(query_embeddings=[query_embedding], n_results=1)
    context = results['documents'][0][0]

    # Step 3: Prompt the LLM with the context
    prompt = f"Using this context: {context}\n\nAnswer this question: {question}"
    
    response = ollama.chat(model='llama3.2:1b', messages=[{'role': 'user', 'content': prompt}])
    return response['message']['content']

if __name__ == "__main__":
    # Run ingestion once (you can comment this out once the DB is built)
    if collection.count() == 0:
        ingest_pdf(PDF_PATH)

    while True:
        query = input("\nAsk about the PDF (or 'exit'): ")
        if query.lower() == 'exit': break
        print("\nAgent:", ask_rag_agent(query))

